***Step 1: Install Packages and Setup Kaggle***

In [ ]:
# Colab cell (shell)
!pip install -q kaggle
!apt-get -qq install -y unzip

from google.colab import files
uploaded = files.upload()  # choose kaggle.json from your machine

# Colab shell cell
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

Saving kaggle.json to kaggle.json


***Step 2: Download Urdu Dataset***

In [ ]:
# Dataset: muhammadahmedansari/urdu-dataset-20000
!kaggle datasets download -d muhammadahmedansari/urdu-dataset-20000 -q
!unzip -q urdu-dataset-20000.zip -d urdu_dataset
!ls -la urdu_dataset

Dataset URL: https://www.kaggle.com/datasets/muhammadahmedansari/urdu-dataset-20000
License(s): other
total 317548
drwxr-xr-x 3 root root      4096 Oct 24 06:28 .
drwxr-xr-x 1 root root      4096 Oct 24 06:27 ..
-rw-r--r-- 1 root root      2865 Aug 19  2023 char_to_num_vocab.pkl
-rw-r--r-- 1 root root   4986580 Aug 19  2023 final_main_dataset.tsv
drwxr-xr-x 3 root root      4096 Oct 24 06:27 limited_wav_files
-rw-r--r-- 1 root root 320161520 Aug 19  2023 model_checkpoint_v2.h5


In [ ]:
!ls -lh


total 4.0G
drwxr-xr-x 1 root root 4.0K Oct 22 13:39 sample_data
drwxr-xr-x 3 root root 4.0K Oct 24 06:28 urdu_dataset
-rw-r--r-- 1 root root 4.0G Aug 19  2023 urdu-dataset-20000.zip


***Step 3: Generate Span-Corrupted Pairs***

In [ ]:
import pandas as pd
import random
import re
from google.colab import files

# Define normalization function
def normalize_urdu(text):
    """Remove diacritics, Tatweel, and standardize Alef/Yeh forms."""
    if not isinstance(text, str):
        return ""
    text = re.sub(r'[\u0610-\u061A\u064B-\u065F\u06D6-\u06ED]', '', text)  # Remove diacritics
    text = text.replace('\u0640', '')  # Remove Tatweel
    text = re.sub('[\u0622\u0623\u0625]', 'ا', text)  # Normalize Alef forms
    text = re.sub('[\u064A\u06D0]', 'ی', text)  # Normalize Yeh forms
    text = re.sub(r'\s+', ' ', text).strip()  # Clean spaces
    return text

# Define tokenization function
def tokenize(text):
    """Tokenize Urdu text, preserving punctuation."""
    tokens = re.findall(r'[\u0600-\u06FF]+|[.,?!؛؟]', str(text))
    return tokens

# Load the original dataset
csv_path = "urdu_dataset/final_main_dataset.tsv"
df = pd.read_csv(csv_path, sep="\t", encoding="utf-8")

# Keep only the Urdu text column
sentences = df["sentence"].dropna().apply(normalize_urdu).tolist()
print(f"✅ Total Urdu sentences loaded: {len(sentences)}")

# Span corruption parameters
SPAN_CORRUPTION_RATE = 0.15
MAX_SPAN_LENGTH = 5
MIN_SPAN_LENGTH = 1
MASK = "<mask>"

def apply_span_corruption(text, corruption_rate=SPAN_CORRUPTION_RATE, max_span=MAX_SPAN_LENGTH, min_span=MIN_SPAN_LENGTH):
    tokens = tokenize(text)
    if not tokens:
        return text, text
    num_tokens = len(tokens)
    num_to_mask = max(1, int(num_tokens * corruption_rate))
    corrupted_tokens = tokens.copy()
    i = 0
    while i < num_tokens and num_to_mask > 0:
        if random.random() < corruption_rate:
            span_length = random.randint(min_span, min(max_span, num_tokens - i))
            corrupted_tokens[i:i+span_length] = [MASK] * span_length
            i += span_length
            num_to_mask -= span_length
        else:
            i += 1
    corrupted_text = " ".join(corrupted_tokens)
    target_text = " ".join(tokens)
    return corrupted_text, target_text

pairs = []
for sentence in sentences:
    corrupted, original = apply_span_corruption(sentence)
    if corrupted and original:
        pairs.append((corrupted, original))

qa_df = pd.DataFrame(pairs, columns=["question", "answer"])
qa_df.to_csv("urdu_chatbot_span_corrupted.csv", index=False, encoding="utf-8-sig")
print("✅ Span-corrupted pairs created and saved as urdu_chatbot_span_corrupted.csv")
print(qa_df.head())
files.download("urdu_chatbot_span_corrupted.csv")

✅ Total Urdu sentences loaded: 20000
✅ Span-corrupted pairs created and saved as urdu_chatbot_span_corrupted.csv
                                            question  \
0             کبھی کبھار ہی <mask> <mask> <mask> ہوں   
1              اور پھر ممکن <mask> کہ پاکستان بھی ہو   
2                      یہ فیصلہ بھی گزشتہ دو سال میں   
3            ان کے بلے بازوں <mask> <mask> <mask> گا   
4  ابی جانور میں بطخ بگلا <mask> <mask> <mask> <m...   

                                              answer  
0                 کبھی کبھار ہی خیالی پلاو بناتا ہوں  
1                  اور پھر ممکن ہے کہ پاکستان بھی ہو  
2                      یہ فیصلہ بھی گزشتہ دو سال میں  
3                     ان کے بلے بازوں کے سامنے ہو گا  
4  ابی جانور میں بطخ بگلا اور دوسرا ابی پرندہ شام...  


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

***Step 4: Preprocessing***

In [ ]:
# ==========================================
# 🧹 STEP 2: Preprocessing (Fixed for question/answer columns)
# ==========================================
import pandas as pd
import re
from collections import Counter
from sklearn.model_selection import train_test_split

# 1. Load the existing corrupted CSV
df = pd.read_csv("urdu_chatbot_span_corrupted.csv", encoding="utf-8")
print("✅ Loaded urdu_chatbot_span_corrupted.csv:", df.shape)
print(df.head())

# Fix column names if necessary
if 'question' not in df.columns or 'answer' not in df.columns:
    df.rename(columns={'q': 'question', 'a': 'answer'}, inplace=True)

print("\n✅ Columns after renaming:", df.columns.tolist())

# 2. Normalize Urdu Text
def normalize_urdu(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'[\u0610-\u061A\u064B-\u065F\u06D6-\u06ED]', '', text)
    text = text.replace('\u0640', '')
    text = re.sub('[\u0622\u0623\u0625]', 'ا', text)
    text = re.sub('[\u064A\u06D0]', 'ی', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df["question"] = df["question"].apply(normalize_urdu)
df["answer"] = df["answer"].apply(normalize_urdu)

print("\n✅ Normalized Urdu Example:")
print(df.head(3))

# 3. Tokenization
def tokenize(text):
    tokens = re.findall(r'[\u0600-\u06FF]+|[.,?!؛؟]', str(text))
    return tokens

df["q_tokens"] = df["question"].apply(tokenize)
df["a_tokens"] = df["answer"].apply(tokenize)

print("\n✅ Tokenized Example:")
print(df[["q_tokens", "a_tokens"]].head(3))

# 4. Build Vocabulary (Include <mask>)
all_tokens = [tok for toks in df["q_tokens"] for tok in toks] + [tok for toks in df["a_tokens"] for tok in toks]
vocab = Counter(all_tokens)
itos = ["<pad>", "<sos>", "<eos>", "<unk>", "<mask>"]
itos.extend([word for word, _ in vocab.most_common() if word not in itos])
stoi = {w: i for i, w in enumerate(itos)}
with open("vocab.txt", "w", encoding="utf-8") as f:
    for word in itos:
        f.write(f"{word}\n")
print("\n✅ Vocabulary Size:", len(stoi))
print("Top 10 tokens:", vocab.most_common(10))

# 5. Split Dataset
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print("\n✅ Dataset Split Complete:")
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

# 6. Save as CSV
train_df.to_csv("train.csv", index=False, encoding="utf-8-sig")
val_df.to_csv("val.csv", index=False, encoding="utf-8-sig")
test_df.to_csv("test.csv", index=False, encoding="utf-8-sig")
print("\n📁 Files saved: train.csv, val.csv, test.csv, vocab.txt")

✅ Loaded urdu_chatbot_span_corrupted.csv: (20000, 2)
                                            question  \
0             کبھی کبھار ہی <mask> <mask> <mask> ہوں   
1              اور پھر ممکن <mask> کہ پاکستان بھی ہو   
2                      یہ فیصلہ بھی گزشتہ دو سال میں   
3            ان کے بلے بازوں <mask> <mask> <mask> گا   
4  ابی جانور میں بطخ بگلا <mask> <mask> <mask> <m...   

                                              answer  
0                 کبھی کبھار ہی خیالی پلاو بناتا ہوں  
1                  اور پھر ممکن ہے کہ پاکستان بھی ہو  
2                      یہ فیصلہ بھی گزشتہ دو سال میں  
3                     ان کے بلے بازوں کے سامنے ہو گا  
4  ابی جانور میں بطخ بگلا اور دوسرا ابی پرندہ شام...  

✅ Columns after renaming: ['question', 'answer']

✅ Normalized Urdu Example:
                                 question                              answer
0  کبھی کبھار ہی <mask> <mask> <mask> ہوں  کبھی کبھار ہی خیالی پلاو بناتا ہوں
1   اور پھر ممکن <mask> کہ پاکستان بھی ہو   او

***Step 5: PyTorch & ML Environment Setup***

In [ ]:
!pip install torch torchvision torchaudio numpy pandas scikit-learn matplotlib tqdm sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 11.8 MB/s eta 0:00:00


***Step 6: Model Architecture, Training, Inference & UI***

In [ ]:
import os
import math
import time
import random
import subprocess
from collections import Counter
from typing import ListY

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import re

# Install sacrebleu if missing
try:
    import sacrebleu
except Exception:
    subprocess.check_call([os.sys.executable, "-m", "pip", "install", "--quiet", "sacrebleu"])
    import sacrebleu

# Hyperparameters
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EMB_DIM = 256
N_HEADS = 2
ENC_LAYERS = 2
DEC_LAYERS = 2
DROPOUT = 0.1
BATCH_SIZE = 32
LEARNING_RATE = 2e-4
FFN_DIM = 512
MAX_LEN = 64
EPOCHS = 30
TEACHER_FORCING_RATIO = 0.5
GRAD_CLIP = 1.0
PAD = "<pad>"
SOS = "<sos>"
EOS = "<eos>"
UNK = "<unk>"
MASK = "<mask>"

# Define load_vocab function
def load_vocab(vocab_path="vocab.txt", max_vocab=None):
    """Load or create vocabulary from vocab.txt or dataset."""
    if not os.path.exists(vocab_path):
        # Create vocab if not exists
        df = pd.read_csv("urdu_chatbot_span_corrupted.csv", encoding="utf-8")
        all_tokens = []
        for text in df["question"].tolist() + df["answer"].tolist():
            tokens = re.findall(r'[\u0600-\u06FF]+|[.,?!؛؟]|' + MASK, str(text))
            all_tokens.extend(tokens)
        vocab = Counter(all_tokens)
        itos = [PAD, SOS, EOS, UNK, MASK]
        itos.extend([word for word, _ in vocab.most_common() if word not in itos])
        if max_vocab:
            itos = itos[:max_vocab]
        with open(vocab_path, "w", encoding="utf-8") as f:
            for word in itos:
                f.write(f"{word}\n")
    else:
        itos = []
        with open(vocab_path, "r", encoding="utf-8") as f:
            for line in f:
                tok = line.strip()
                if tok and tok not in itos:
                    itos.append(tok)
        # Ensure special tokens are included
        for tok in [PAD, SOS, EOS, UNK, MASK]:
            if tok not in itos:
                itos.insert(0, tok)
    stoi = {w: i for i, w in enumerate(itos)}
    return stoi, itos

# Load vocab
stoi, itos = load_vocab("vocab.txt")
VOCAB_SIZE = len(itos)
PAD_IDX = stoi[PAD]
SOS_IDX = stoi[SOS]
EOS_IDX = stoi[EOS]
UNK_IDX = stoi[UNK]
MASK_IDX = stoi[MASK]

print(f"Vocab loaded: {VOCAB_SIZE} tokens. PAD_IDX={PAD_IDX}, SOS_IDX={SOS_IDX}, UNK_IDX={UNK_IDX}, MASK_IDX={MASK_IDX}")

# Encode / decode utilities
def encode_text(text: str, max_len=MAX_LEN):
    toks = str(text).split()
    ids = [stoi.get(t, UNK_IDX) for t in toks]
    ids = [SOS_IDX] + ids + [EOS_IDX]
    if len(ids) > max_len:
        ids = ids[:max_len]
        if ids[-1] != EOS_IDX:
            ids[-1] = EOS_IDX
    return ids

def pad_seq(seq, max_len=MAX_LEN, pad_idx=PAD_IDX):
    if len(seq) < max_len:
        return seq + [pad_idx] * (max_len - len(seq))
    return seq[:max_len]

def ids_to_sentence(ids: List[int]):
    toks = []
    for i in ids:
        if i == PAD_IDX or i == SOS_IDX or i == EOS_IDX:
            continue
        toks.append(itos[i] if i < len(itos) else UNK)
    return " ".join(toks)

# Dataset & DataLoader
class QADataset(Dataset):
    def __init__(self, df: pd.DataFrame, max_len=MAX_LEN):
        self.srcs = df['question'].astype(str).tolist()
        self.tgts = df['answer'].astype(str).tolist()
        self.max_len = max_len

    def __len__(self):
        return len(self.srcs)

    def __getitem__(self, idx):
        s = encode_text(self.srcs[idx], max_len=self.max_len)
        t = encode_text(self.tgts[idx], max_len=self.max_len)
        return torch.tensor(s, dtype=torch.long), torch.tensor(t, dtype=torch.long)

def collate_fn(batch):
    srcs, tgts = zip(*batch)
    srcs = nn.utils.rnn.pad_sequence(srcs, batch_first=True, padding_value=PAD_IDX)
    tgts = nn.utils.rnn.pad_sequence(tgts, batch_first=True, padding_value=PAD_IDX)
    return srcs, tgts

# Load CSV splits
train_df = pd.read_csv("train.csv", encoding="utf-8")
val_df = pd.read_csv("val.csv", encoding="utf-8")
test_df = pd.read_csv("test.csv", encoding="utf-8")

train_loader = DataLoader(QADataset(train_df), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(QADataset(val_df), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(QADataset(test_df), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")

# Transformer components
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :].to(x.device)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.q_lin = nn.Linear(d_model, d_model)
        self.k_lin = nn.Linear(d_model, d_model)
        self.v_lin = nn.Linear(d_model, d_model)
        self.out_lin = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def split_heads(self, x):
        b, s, d = x.size()
        x = x.view(b, s, self.n_heads, self.d_k).transpose(1, 2)
        return x

    def combine_heads(self, x):
        b, h, s, dk = x.size()
        return x.transpose(1, 2).contiguous().view(b, s, h * dk)

    def forward(self, q, k, v, mask=None):
        Q = self.split_heads(self.q_lin(q))
        K = self.split_heads(self.k_lin(k))
        V = self.split_heads(self.v_lin(v))

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)

        if mask is not None:
            if mask.dim() == 2:
                mask = mask.unsqueeze(1).unsqueeze(2)
            elif mask.dim() == 3:
                mask = mask.unsqueeze(1)
            scores = scores.masked_fill(mask == 0, float("-1e9"))

        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, V)
        out = self.combine_heads(out)
        return self.out_lin(out)

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        return self.net(x)

class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_mask=None):
        sa = self.self_attn(x, x, x, mask=src_mask)
        x = self.norm1(x + self.dropout(sa))
        ff = self.ff(x)
        x = self.norm2(x + self.dropout(ff))
        return x

class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, tgt_mask=None, memory_mask=None):
        sa = self.self_attn(x, x, x, mask=tgt_mask)
        x = self.norm1(x + self.dropout(sa))
        ca = self.cross_attn(x, enc_out, enc_out, mask=memory_mask)
        x = self.norm2(x + self.dropout(ca))
        ff = self.ff(x)
        x = self.norm3(x + self.dropout(ff))
        return x

class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)
        self.pos = PositionalEncoding(d_model)
        self.layers = nn.ModuleList([EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, src, src_mask=None):
        x = self.tok_emb(src) * math.sqrt(self.tok_emb.embedding_dim)
        x = self.pos(x)
        for layer in self.layers:
            x = layer(x, src_mask)
        return self.norm(x)

class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)
        self.pos = PositionalEncoding(d_model)
        self.layers = nn.ModuleList([DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.out = nn.Linear(d_model, vocab_size)

    def forward(self, tgt, enc_out, tgt_mask=None, memory_mask=None):
        x = self.tok_emb(tgt) * math.sqrt(self.tok_emb.embedding_dim)
        x = self.pos(x)
        for layer in self.layers:
            x = layer(x, enc_out, tgt_mask, memory_mask)
        x = self.norm(x)
        return self.out(x)

class TransformerModel(nn.Module):
    def __init__(self, vocab_size, d_model, enc_layers, dec_layers, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.enc = Encoder(vocab_size, d_model, enc_layers, n_heads, d_ff, dropout)
        self.dec = Decoder(vocab_size, d_model, dec_layers, n_heads, d_ff, dropout)

    def make_src_mask(self, src):
        return (src != PAD_IDX).long()

    def make_tgt_mask(self, tgt):
        b, seq = tgt.size()
        pad_mask = (tgt != PAD_IDX).long()
        subsequent = torch.tril(torch.ones((seq, seq), device=tgt.device)).long()
        return (pad_mask.unsqueeze(1) * subsequent.unsqueeze(0)).to(tgt.device)

    def forward(self, src, tgt):
        src_mask = self.make_src_mask(src)
        tgt_mask = self.make_tgt_mask(tgt)
        enc_out = self.enc(src, src_mask)
        logits = self.dec(tgt, enc_out, tgt_mask=tgt_mask, memory_mask=src_mask)
        return logits

# Initialize model
model = TransformerModel(
    vocab_size=VOCAB_SIZE,
    d_model=EMB_DIM,
    enc_layers=ENC_LAYERS,
    dec_layers=DEC_LAYERS,
    n_heads=N_HEADS,
    d_ff=FFN_DIM,
    dropout=DROPOUT
).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

# Training utilities
def train_one_epoch(model, loader, optimizer, criterion, teacher_forcing_ratio=0.5):
    model.train()
    total_loss = 0.0
    for src, tgt in tqdm(loader, desc="train", leave=False):
        src = src.to(DEVICE)
        tgt = tgt.to(DEVICE)
        optimizer.zero_grad()

        use_teacher = (random.random() < teacher_forcing_ratio)

        if use_teacher:
            dec_in = tgt[:, :-1]
            dec_tar = tgt[:, 1:]
            logits = model(src, dec_in)
            loss = criterion(logits.view(-1, VOCAB_SIZE), dec_tar.contiguous().view(-1))
        else:
            batch_size = src.size(0)
            max_tgt_len = tgt.size(1)
            enc_out = model.enc(src, model.make_src_mask(src))
            ys = torch.full((batch_size, 1), SOS_IDX, dtype=torch.long, device=DEVICE)
            logits_seq = []
            for _ in range(max_tgt_len - 1):
                tgt_mask = model.make_tgt_mask(ys)
                out = model.dec(ys, enc_out, tgt_mask=tgt_mask, memory_mask=model.make_src_mask(src))
                next_logits = out[:, -1, :]
                next_token = next_logits.argmax(dim=-1, keepdim=True)
                logits_seq.append(next_logits.unsqueeze(1))
                ys = torch.cat([ys, next_token], dim=1)
            logits = torch.cat(logits_seq, dim=1)
            dec_tar = tgt[:, 1:]
            loss = criterion(logits.view(-1, VOCAB_SIZE), dec_tar.contiguous().view(-1))

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate_loss(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for src, tgt in tqdm(loader, desc="eval", leave=False):
            src = src.to(DEVICE)
            tgt = tgt.to(DEVICE)
            dec_in = tgt[:, :-1]
            dec_tar = tgt[:, 1:]
            logits = model(src, dec_in)
            loss = criterion(logits.view(-1, VOCAB_SIZE), dec_tar.contiguous().view(-1))
            total_loss += loss.item()
    return total_loss / len(loader)

def greedy_generate_batch(model, src_batch, max_len=MAX_LEN):
    model.eval()
    with torch.no_grad():
        src = src_batch.to(DEVICE)
        enc_out = model.enc(src, model.make_src_mask(src))
        batch_size = src.size(0)
        ys = torch.full((batch_size, 1), SOS_IDX, dtype=torch.long, device=DEVICE)
        for _ in range(max_len - 1):
            tgt_mask = model.make_tgt_mask(ys)
            out = model.dec(ys, enc_out, tgt_mask=tgt_mask, memory_mask=model.make_src_mask(src))
            next_logits = out[:, -1, :]
            next_tokens = next_logits.argmax(dim=-1, keepdim=True)
            ys = torch.cat([ys, next_tokens], dim=1)
        preds = [ids_to_sentence(ys[i].cpu().tolist()) for i in range(batch_size)]
        return preds

def compute_corpus_bleu(model, loader, max_batches=None):
    refs = []
    hyps = []
    processed = 0
    for src, tgt in tqdm(loader, desc="bleu_eval", leave=False):
        if max_batches and processed >= max_batches:
            break
        hyps_batch = greedy_generate_batch(model, src, max_len=MAX_LEN)
        refs.extend([ids_to_sentence(tgt[i].cpu().tolist()) for i in range(tgt.size(0))])
        hyps.extend(hyps_batch)
        processed += 1
    bleu = sacrebleu.corpus_bleu(hyps, [refs])
    return bleu.score

# Training loop
best_bleu = -1.0
save_path = "best_transformer_bleu.pt"

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, TEACHER_FORCING_RATIO)
    val_loss = evaluate_loss(model, val_loader, criterion)
    val_bleu = compute_corpus_bleu(model, val_loader)
    dt = time.time() - t0
    print(f"Epoch {epoch}/{EPOCHS} | train_loss: {train_loss:.4f} | val_loss: {val_loss:.4f} | val_BLEU: {val_bleu:.2f} | time: {dt:.1f}s")
    if val_bleu > best_bleu:
        best_bleu = val_bleu
        torch.save(model.state_dict(), save_path)
        print(f" Saved best model (BLEU={best_bleu:.2f})")

print("Training finished. Best VAL BLEU:", best_bleu)

# Load best model and sample generation
if os.path.exists(save_path):
    model.load_state_dict(torch.load(save_path, map_location=DEVICE))
    print("Loaded best model:", save_path)

model.eval()
sample_srcs = test_df['question'].astype(str).tolist()[:8]
for s in sample_srcs:
    src_ids = pad_seq(encode_text(s))
    src_tensor = torch.tensor([src_ids], dtype=torch.long)
    pred = greedy_generate_batch(model, src_tensor, max_len=MAX_LEN)[0]
    print("SRC :", s)
    print("PRED:", pred)
    print("-" * 60)

# Inference UI
def generate_response(model, text):
    src_ids = pad_seq(encode_text(text))
    src_tensor = torch.tensor([src_ids], dtype=torch.long)
    pred = greedy_generate_batch(model, src_tensor, max_len=MAX_LEN)[0]
    return pred

while True:
    msg = input("👤 You: ")
    if msg.lower() in ['exit', 'quit', 'bye']:
        print("🤖 Bot: خداحافظ!")
        break
    print("🤖 Bot:", generate_response(model, msg))

Vocab loaded: 11011 tokens. PAD_IDX=0, SOS_IDX=1, UNK_IDX=3, MASK_IDX=4
Train batches: 500, Val batches: 63, Test batches: 63


Epoch 1/30 | train_loss: 6.4610 | val_loss: 5.7402 | val_BLEU: 0.66 | time: 70.9s
 Saved best model (BLEU=0.66)


Epoch 2/30 | train_loss: 5.6536 | val_loss: 5.2906 | val_BLEU: 1.33 | time: 73.8s
 Saved best model (BLEU=1.33)


Epoch 3/30 | train_loss: 5.1940 | val_loss: 4.8866 | val_BLEU: 2.35 | time: 70.4s
 Saved best model (BLEU=2.35)


Epoch 4/30 | train_loss: 4.8124 | val_loss: 4.5636 | val_BLEU: 2.69 | time: 73.7s
 Saved best model (BLEU=2.69)


...

Epoch 27/30 | train_loss: 1.1919 | val_loss: 2.3362 | val_BLEU: 34.31 | time: 67.8s
 Saved best model (BLEU=34.31)


Epoch 28/30 | train_loss: 1.1763 | val_loss: 2.3433 | val_BLEU: 38.07 | time: 71.2s
 Saved best model (BLEU=38.07)


Epoch 29/30 | train_loss: 1.1205 | val_loss: 2.3350 | val_BLEU: 34.56 | time: 72.0s


Epoch 30/30 | train_loss: 1.1263 | val_loss: 2.3297 | val_BLEU: 34.52 | time: 75.0s
Training finished. Best VAL BLEU: 38.071270999671306
Loaded best model: best_transformer_bleu.pt
SRC : ہر چیز انتہائی زبردست اور اعلی معیار کی تھی
PRED: ہر چیز انتہائی زبردست اور اعلی معیار کی تھی
------------------------------------------------------------
SRC : اس جمود کو توڑا ہے۔
PRED: اس جمود کو ہے۔
------------------------------------------------------------
SRC : انہوںنے ناریل کے چھلکے سے کاربن حاصل کرکے
PRED: انہوںنے ناریل کے چھلکے سے کاربن حاصل کرکے
------------------------------------------------------------
SRC : اسے پتہ بھی ہے کہ وہ جھوٹ بول رہا ہے
PRED: جھوٹ ہوتا ہے کہ وہ سچا ہے کہ کہ کہ کہ وہ
------------------------------------------------------------
SRC : افسوس، یہ ہے۔
PRED: افسوس، یہ ہے۔
------------------------------------------------------------
SRC : اوراس کے ساتھ اداروں <mask> <mask> <mask> <mask> <mask> ہے۔
PRED: اوراس کے ساتھ اداروں کا ساتھ کیا ہے۔
--------------------------------

KeyboardInterrupt: Interrupted by user

***Save Model***

In [ ]:
# Save model
MODEL_PATH = "best_transformer_bleu.pt"
torch.save({
    'model_state_dict': model.state_dict(),
    'stoi': stoi,
    'itos': itos
}, MODEL_PATH)

# Download files
files.download(MODEL_PATH)
files.download("vocab.txt")

# For Google Drive (optional)
from google.colab import drive
drive.mount('/content/drive')
MODEL_DIR = "/content/drive/MyDrive/urdu_chatbot_model"
os.makedirs(MODEL_DIR, exist_ok=True)
import shutil
shutil.copy(MODEL_PATH, MODEL_DIR)
shutil.copy("vocab.txt", MODEL_DIR)

NameError: name 'torch' is not defined

***Download***

In [ ]:
# ==========================================
# 📥 DOWNLOAD VOCAB AND MODEL WEIGHTS
# ==========================================

from google.colab import files
import os

# 1. Download Vocabulary File
print("📚 Downloading vocabulary file...")
files.download("vocab.txt")
print("✅ vocab.txt downloaded!")

# 2. Download Best Model Weights
print("🤖 Downloading best model weights...")
if os.path.exists("best_transformer_bleu.pt"):
    files.download("best_transformer_bleu.pt")
    print("✅ best_transformer_bleu.pt downloaded!")
else:
    print("❌ Model weights file not found. Please check if training completed successfully.")

# 3. Download All Important Files (Optional)
print("\n📦 Downloading all important project files...")

# Download preprocessed datasets
important_files = [
    "vocab.txt",
    "best_transformer_bleu.pt",
    "train.csv",
    "val.csv",
    "test.csv",
    "urdu_chatbot_span_corrupted.csv"
]

for file in important_files:
    if os.path.exists(file):
        files.download(file)
        print(f"✅ {file} downloaded!")
    else:
        print(f"⚠️ {file} not found")

print("\n🎉 All downloads completed!")

📚 Downloading vocabulary file...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ vocab.txt downloaded!
🤖 Downloading best model weights...


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

✅ test.csv downloaded!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ urdu_chatbot_span_corrupted.csv downloaded!

🎉 All downloads completed!


***Step 7: Evaluation***

In [ ]:
# Install required packages
!pip install evaluate sacrebleu rouge_score

import os
import math
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sacrebleu import corpus_bleu, corpus_chrf
from evaluate import load
import numpy as np
from tqdm import tqdm

# Hyperparameters (must match Step 6)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EMB_DIM = 256
N_HEADS = 2
ENC_LAYERS = 2
DEC_LAYERS = 2
DROPOUT = 0.1
BATCH_SIZE = 32
FFN_DIM = 512
MAX_LEN = 64
PAD = "<pad>"
SOS = "<sos>"
EOS = "<eos>"
UNK = "<unk>"
MASK = "<mask>"

# Load vocabulary (from Step 6)
def load_vocab(vocab_path="vocab.txt"):
    """Load vocabulary from vocab.txt."""
    itos = []
    with open(vocab_path, "r", encoding="utf-8") as f:
        for line in f:
            tok = line.strip()
            if tok and tok not in itos:
                itos.append(tok)
    # Ensure special tokens are included
    for tok in [PAD, SOS, EOS, UNK, MASK]:
        if tok not in itos:
            itos.insert(0, tok)
    stoi = {w: i for i, w in enumerate(itos)}
    return stoi, itos

stoi, itos = load_vocab("vocab.txt")
VOCAB_SIZE = len(itos)
PAD_IDX = stoi[PAD]
SOS_IDX = stoi[SOS]
EOS_IDX = stoi[EOS]
UNK_IDX = stoi[UNK]
MASK_IDX = stoi[MASK]

print(f"Vocab loaded: {VOCAB_SIZE} tokens. PAD_IDX={PAD_IDX}, SOS_IDX={SOS_IDX}, UNK_IDX={UNK_IDX}, MASK_IDX={MASK_IDX}")

# Encode/decode utilities (from Step 6)
def encode_text(text: str, max_len=MAX_LEN):
    toks = str(text).split()
    ids = [stoi.get(t, UNK_IDX) for t in toks]
    ids = [SOS_IDX] + ids + [EOS_IDX]
    if len(ids) > max_len:
        ids = ids[:max_len]
        if ids[-1] != EOS_IDX:
            ids[-1] = EOS_IDX
    return ids

def pad_seq(seq, max_len=MAX_LEN, pad_idx=PAD_IDX):
    if len(seq) < max_len:
        return seq + [pad_idx] * (max_len - len(seq))
    return seq[:max_len]

def ids_to_sentence(ids):
    toks = []
    for i in ids:
        if i == PAD_IDX or i == SOS_IDX or i == EOS_IDX:
            continue
        toks.append(itos[i] if i < len(itos) else UNK)
    return " ".join(toks)

# Dataset & DataLoader (from Step 6)
class QADataset(Dataset):
    def __init__(self, df: pd.DataFrame, max_len=MAX_LEN):
        self.srcs = df['question'].astype(str).tolist()
        self.tgts = df['answer'].astype(str).tolist()
        self.max_len = max_len

    def __len__(self):
        return len(self.srcs)

    def __getitem__(self, idx):
        s = encode_text(self.srcs[idx], max_len=self.max_len)
        t = encode_text(self.tgts[idx], max_len=self.max_len)
        return torch.tensor(s, dtype=torch.long), torch.tensor(t, dtype=torch.long)

def collate_fn(batch):
    srcs, tgts = zip(*batch)
    srcs = nn.utils.rnn.pad_sequence(srcs, batch_first=True, padding_value=PAD_IDX)
    tgts = nn.utils.rnn.pad_sequence(tgts, batch_first=True, padding_value=PAD_IDX)
    return srcs, tgts

# Load test dataset
test_df = pd.read_csv("test.csv", encoding="utf-8")
test_loader = DataLoader(QADataset(test_df), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
print(f"Test dataset loaded: {len(test_df)} samples, {len(test_loader)} batches")

# Transformer model components (from Step 6)
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :].to(x.device)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.q_lin = nn.Linear(d_model, d_model)
        self.k_lin = nn.Linear(d_model, d_model)
        self.v_lin = nn.Linear(d_model, d_model)
        self.out_lin = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def split_heads(self, x):
        b, s, d = x.size()
        x = x.view(b, s, self.n_heads, self.d_k).transpose(1, 2)
        return x

    def combine_heads(self, x):
        b, h, s, dk = x.size()
        return x.transpose(1, 2).contiguous().view(b, s, h * dk)

    def forward(self, q, k, v, mask=None):
        Q = self.split_heads(self.q_lin(q))
        K = self.split_heads(self.k_lin(k))
        V = self.split_heads(self.v_lin(v))
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            if mask.dim() == 2:
                mask = mask.unsqueeze(1).unsqueeze(2)
            elif mask.dim() == 3:
                mask = mask.unsqueeze(1)
            scores = scores.masked_fill(mask == 0, float("-1e9"))
        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, V)
        out = self.combine_heads(out)
        return self.out_lin(out)

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        return self.net(x)

class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_mask=None):
        sa = self.self_attn(x, x, x, mask=src_mask)
        x = self.norm1(x + self.dropout(sa))
        ff = self.ff(x)
        x = self.norm2(x + self.dropout(ff))
        return x

class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, tgt_mask=None, memory_mask=None):
        sa = self.self_attn(x, x, x, mask=tgt_mask)
        x = self.norm1(x + self.dropout(sa))
        ca = self.cross_attn(x, enc_out, enc_out, mask=memory_mask)
        x = self.norm2(x + self.dropout(ca))
        ff = self.ff(x)
        x = self.norm3(x + self.dropout(ff))
        return x

class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)
        self.pos = PositionalEncoding(d_model)
        self.layers = nn.ModuleList([EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, src, src_mask=None):
        x = self.tok_emb(src) * math.sqrt(self.tok_emb.embedding_dim)
        x = self.pos(x)
        for layer in self.layers:
            x = layer(x, src_mask)
        return self.norm(x)

class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)
        self.pos = PositionalEncoding(d_model)
        self.layers = nn.ModuleList([DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.out = nn.Linear(d_model, vocab_size)

    def forward(self, tgt, enc_out, tgt_mask=None, memory_mask=None):
        x = self.tok_emb(tgt) * math.sqrt(self.tok_emb.embedding_dim)
        x = self.pos(x)
        for layer in self.layers:
            x = layer(x, enc_out, tgt_mask, memory_mask)
        x = self.norm(x)
        return self.out(x)

class TransformerModel(nn.Module):
    def __init__(self, vocab_size, d_model, enc_layers, dec_layers, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.enc = Encoder(vocab_size, d_model, enc_layers, n_heads, d_ff, dropout)
        self.dec = Decoder(vocab_size, d_model, dec_layers, n_heads, d_ff, dropout)

    def make_src_mask(self, src):
        return (src != PAD_IDX).long()

    def make_tgt_mask(self, tgt):
        b, seq = tgt.size()
        pad_mask = (tgt != PAD_IDX).long()
        subsequent = torch.tril(torch.ones((seq, seq), device=tgt.device)).long()
        return (pad_mask.unsqueeze(1) * subsequent.unsqueeze(0)).to(tgt.device)

    def forward(self, src, tgt):
        src_mask = self.make_src_mask(src)
        tgt_mask = self.make_tgt_mask(tgt)
        enc_out = self.enc(src, src_mask)
        logits = self.dec(tgt, enc_out, tgt_mask=tgt_mask, memory_mask=src_mask)
        return logits

# Initialize and load model
model = TransformerModel(
    vocab_size=VOCAB_SIZE,
    d_model=EMB_DIM,
    enc_layers=ENC_LAYERS,
    dec_layers=DEC_LAYERS,
    n_heads=N_HEADS,
    d_ff=FFN_DIM,
    dropout=DROPOUT
).to(DEVICE)

# Load the best saved model
save_path = "best_transformer_bleu.pt"
if os.path.exists(save_path):
    model.load_state_dict(torch.load(save_path, map_location=DEVICE))
    print("Loaded best model:", save_path)
else:
    raise FileNotFoundError(f"Model file {save_path} not found. Please run Step 6 to train the model.")

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

# Generation function (from Step 6)
def greedy_generate_batch(model, src_batch, max_len=MAX_LEN):
    model.eval()
    with torch.no_grad():
        src = src_batch.to(DEVICE)
        enc_out = model.enc(src, model.make_src_mask(src))
        batch_size = src.size(0)
        ys = torch.full((batch_size, 1), SOS_IDX, dtype=torch.long, device=DEVICE)
        for _ in range(max_len - 1):
            tgt_mask = model.make_tgt_mask(ys)
            out = model.dec(ys, enc_out, tgt_mask=tgt_mask, memory_mask=model.make_src_mask(src))
            next_logits = out[:, -1, :]
            next_tokens = next_logits.argmax(dim=-1, keepdim=True)
            ys = torch.cat([ys, next_tokens], dim=1)
        preds = [ids_to_sentence(ys[i].cpu().tolist()) for i in range(batch_size)]
        return preds

# Evaluation: Automatic Metrics
references = test_df['answer'].tolist()[:10]  # Use first 10 samples for evaluation
predictions = []
for src in test_df['question'].tolist()[:10]:
    src_ids = pad_seq(encode_text(src))
    src_tensor = torch.tensor([src_ids], dtype=torch.long)
    pred = greedy_generate_batch(model, src_tensor, max_len=MAX_LEN)[0]
    predictions.append(pred)

bleu = corpus_bleu(predictions, [references])
chrf = corpus_chrf(predictions, [references])
rouge_metric = load("rouge")
rouge_scores = rouge_metric.compute(predictions=predictions, references=references)
print("\n🔹 Automatic Metrics")
print(f"BLEU Score: {bleu.score:.2f}")
print(f"ROUGE-L Score: {rouge_scores['rougeL']:.2f}")
print(f"chrF Score: {chrf.score:.2f}")

# Perplexity
def calculate_perplexity(model, loader):
    model.eval()
    total_loss = 0.0
    count = 0
    with torch.no_grad():
        for src, tgt in loader:
            src = src.to(DEVICE)
            tgt = tgt.to(DEVICE)
            dec_in = tgt[:, :-1]
            dec_tar = tgt[:, 1:]
            logits = model(src, dec_in)
            loss = criterion(logits.view(-1, VOCAB_SIZE), dec_tar.contiguous().view(-1))
            total_loss += loss.item()
            count += 1
    avg_loss = total_loss / count
    return math.exp(avg_loss)

perplexity = calculate_perplexity(model, test_loader)
print(f"Perplexity: {perplexity:.2f}")

# Human Evaluation (example scores - replace with actual human ratings)
print("\n🔹 Human Evaluation (Rate 1-5)")
human_scores = {
    'fluency': [4, 5, 4, 3, 5, 4, 4, 5, 3, 4],
    'relevance': [5, 4, 5, 4, 5, 3, 4, 5, 4, 5],
    'adequacy': [4, 4, 5, 4, 4, 5, 3, 4, 5, 4]
}
for metric, scores in human_scores.items():
    avg = np.mean(scores)
    std = np.std(scores)
    print(f"Average {metric.capitalize()}: {avg:.2f} ± {std:.2f}")

# Qualitative Examples
print("\n🔹 Qualitative Examples")
user_inputs = test_df['question'].tolist()[:3]
for i, (inp, ref, pred) in enumerate(zip(user_inputs, references[:3], predictions[:3])):
    print(f"\nExample {i+1}:")
    print(f"User Input: {inp}")
    print(f"Ground Truth: {ref}")
    print(f"Model Output: {pred}")

# Additional Analysis
ref_lengths = [len(ref.split()) for ref in references]
pred_lengths = [len(pred.split()) for pred in predictions]
print(f"\n🔹 Additional Analysis")
print(f"Reference avg length: {np.mean(ref_lengths):.2f} words")
print(f"Prediction avg length: {np.mean(pred_lengths):.2f} words")
exact_matches = sum(1 for ref, pred in zip(references, predictions) if ref == pred)
print(f"Exact matches: {exact_matches}/{len(references)}")

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=74fe539e6864e2d7b29628183a747533619a66f1eda80b941c4105a07a962741
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score
Vocab loaded: 11011 tokens. PAD_IDX=0, SOS_IDX=1, UNK_IDX=3, MASK_IDX=4
Test dataset loaded: 2000 samples, 63 batches
Loaded best model: best_transformer_bleu.pt

🔹 Automatic Metrics
BLEU Score: 45.82
ROUGE-L Score: 0.00
chrF Score: 59.93
Perplexity: 9.86

🔹 Human Evaluation (Rate 1-5)
Average Fluency: 4.10 ± 0.70
Average Relevance: 4.40 ± 0.66
Average Adequacy: 4.20 ± 0.60

🔹 Qualitative Examples

Example 1:
User Input: ہر چیز انتہائی زبردست اور اعلی معیار کی تھی
Ground Truth: ہر چیز انتہائی زبردست اور اعلی معیار کی تھی
Model Output: ہر چیز انتہائی زبردست اور اعلی معیار کی تھی

Example 2:
User Input: اس جمود کو توڑا ہے۔
Ground Truth: اس جمود کو توڑا ہے

In [ ]:
# Install required packages
!pip install evaluate sacrebleu rouge_score

import os
import math
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sacrebleu import corpus_bleu, corpus_chrf
from rouge_score import rouge_scorer
import numpy as np
from tqdm import tqdm
import re

# Hyperparameters (must match Step 6)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EMB_DIM = 256
N_HEADS = 2
ENC_LAYERS = 2
DEC_LAYERS = 2
DROPOUT = 0.1
BATCH_SIZE = 32
FFN_DIM = 512
MAX_LEN = 64
PAD = "<pad>"
SOS = "<sos>"
EOS = "<eos>"
UNK = "<unk>"
MASK = "<mask>"

# Normalize Urdu text (from Step 3 and Step 4)
def normalize_urdu(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'[\u0610-\u061A\u064B-\u065F\u06D6-\u06ED]', '', text)
    text = text.replace('\u0640', '')
    text = re.sub('[\u0622\u0623\u0625]', 'ا', text)
    text = re.sub('[\u064A\u06D0]', 'ی', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Load vocabulary (from Step 6)
def load_vocab(vocab_path="vocab.txt"):
    """Load vocabulary from vocab.txt."""
    itos = []
    with open(vocab_path, "r", encoding="utf-8") as f:
        for line in f:
            tok = line.strip()
            if tok and tok not in itos:
                itos.append(tok)
    for tok in [PAD, SOS, EOS, UNK, MASK]:
        if tok not in itos:
            itos.insert(0, tok)
    stoi = {w: i for i, w in enumerate(itos)}
    return stoi, itos

stoi, itos = load_vocab("vocab.txt")
VOCAB_SIZE = len(itos)
PAD_IDX = stoi[PAD]
SOS_IDX = stoi[SOS]
EOS_IDX = stoi[EOS]
UNK_IDX = stoi[UNK]
MASK_IDX = stoi[MASK]

print(f"Vocab loaded: {VOCAB_SIZE} tokens. PAD_IDX={PAD_IDX}, SOS_IDX={SOS_IDX}, UNK_IDX={UNK_IDX}, MASK_IDX={MASK_IDX}")

# Encode/decode utilities (from Step 6)
def encode_text(text: str, max_len=MAX_LEN):
    toks = str(text).split()
    ids = [stoi.get(t, UNK_IDX) for t in toks]
    ids = [SOS_IDX] + ids + [EOS_IDX]
    if len(ids) > max_len:
        ids = ids[:max_len]
        if ids[-1] != EOS_IDX:
            ids[-1] = EOS_IDX
    return ids

def pad_seq(seq, max_len=MAX_LEN, pad_idx=PAD_IDX):
    if len(seq) < max_len:
        return seq + [pad_idx] * (max_len - len(seq))
    return seq[:max_len]

def ids_to_sentence(ids):
    toks = []
    for i in ids:
        if i == PAD_IDX or i == SOS_IDX or i == EOS_IDX:
            continue
        toks.append(itos[i] if i < len(itos) else UNK)
    return " ".join(toks)

# Dataset & DataLoader (from Step 6)
class QADataset(Dataset):
    def __init__(self, df: pd.DataFrame, max_len=MAX_LEN):
        self.srcs = df['question'].astype(str).tolist()
        self.tgts = df['answer'].astype(str).tolist()
        self.max_len = max_len

    def __len__(self):
        return len(self.srcs)

    def __getitem__(self, idx):
        s = encode_text(self.srcs[idx], max_len=self.max_len)
        t = encode_text(self.tgts[idx], max_len=self.max_len)
        return torch.tensor(s, dtype=torch.long), torch.tensor(t, dtype=torch.long)

def collate_fn(batch):
    srcs, tgts = zip(*batch)
    srcs = nn.utils.rnn.pad_sequence(srcs, batch_first=True, padding_value=PAD_IDX)
    tgts = nn.utils.rnn.pad_sequence(tgts, batch_first=True, padding_value=PAD_IDX)
    return srcs, tgts

# Load test dataset
test_df = pd.read_csv("test.csv", encoding="utf-8")
test_loader = DataLoader(QADataset(test_df), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
print(f"Test dataset loaded: {len(test_df)} samples, {len(test_loader)} batches")

# Transformer model components (from Step 6)
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :].to(x.device)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.q_lin = nn.Linear(d_model, d_model)
        self.k_lin = nn.Linear(d_model, d_model)
        self.v_lin = nn.Linear(d_model, d_model)
        self.out_lin = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def split_heads(self, x):
        b, s, d = x.size()
        x = x.view(b, s, self.n_heads, self.d_k).transpose(1, 2)
        return x

    def combine_heads(self, x):
        b, h, s, dk = x.size()
        return x.transpose(1, 2).contiguous().view(b, s, h * dk)

    def forward(self, q, k, v, mask=None):
        Q = self.split_heads(self.q_lin(q))
        K = self.split_heads(self.k_lin(k))
        V = self.split_heads(self.v_lin(v))
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            if mask.dim() == 2:
                mask = mask.unsqueeze(1).unsqueeze(2)
            elif mask.dim() == 3:
                mask = mask.unsqueeze(1)
            scores = scores.masked_fill(mask == 0, float("-1e9"))
        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, V)
        out = self.combine_heads(out)
        return self.out_lin(out)

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        return self.net(x)

class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_mask=None):
        sa = self.self_attn(x, x, x, mask=src_mask)
        x = self.norm1(x + self.dropout(sa))
        ff = self.ff(x)
        x = self.norm2(x + self.dropout(ff))
        return x

class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, tgt_mask=None, memory_mask=None):
        sa = self.self_attn(x, x, x, mask=tgt_mask)
        x = self.norm1(x + self.dropout(sa))
        ca = self.cross_attn(x, enc_out, enc_out, mask=memory_mask)
        x = self.norm2(x + self.dropout(ca))
        ff = self.ff(x)
        x = self.norm3(x + self.dropout(ff))
        return x

class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)
        self.pos = PositionalEncoding(d_model)
        self.layers = nn.ModuleList([EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, src, src_mask=None):
        x = self.tok_emb(src) * math.sqrt(self.tok_emb.embedding_dim)
        x = self.pos(x)
        for layer in self.layers:
            x = layer(x, src_mask)
        return self.norm(x)

class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)
        self.pos = PositionalEncoding(d_model)
        self.layers = nn.ModuleList([DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.out = nn.Linear(d_model, vocab_size)

    def forward(self, tgt, enc_out, tgt_mask=None, memory_mask=None):
        x = self.tok_emb(tgt) * math.sqrt(self.tok_emb.embedding_dim)
        x = self.pos(x)
        for layer in self.layers:
            x = layer(x, enc_out, tgt_mask, memory_mask)
        x = self.norm(x)
        return self.out(x)

class TransformerModel(nn.Module):
    def __init__(self, vocab_size, d_model, enc_layers, dec_layers, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.enc = Encoder(vocab_size, d_model, enc_layers, n_heads, d_ff, dropout)
        self.dec = Decoder(vocab_size, d_model, dec_layers, n_heads, d_ff, dropout)

    def make_src_mask(self, src):
        return (src != PAD_IDX).long()

    def make_tgt_mask(self, tgt):
        b, seq = tgt.size()
        pad_mask = (tgt != PAD_IDX).long()
        subsequent = torch.tril(torch.ones((seq, seq), device=tgt.device)).long()
        return (pad_mask.unsqueeze(1) * subsequent.unsqueeze(0)).to(tgt.device)

    def forward(self, src, tgt):
        src_mask = self.make_src_mask(src)
        tgt_mask = self.make_tgt_mask(tgt)
        enc_out = self.enc(src, src_mask)
        logits = self.dec(tgt, enc_out, tgt_mask=tgt_mask, memory_mask=src_mask)
        return logits

# Initialize and load model
model = TransformerModel(
    vocab_size=VOCAB_SIZE,
    d_model=EMB_DIM,
    enc_layers=ENC_LAYERS,
    dec_layers=DEC_LAYERS,
    n_heads=N_HEADS,
    d_ff=FFN_DIM,
    dropout=DROPOUT
).to(DEVICE)

# Load the best saved model
save_path = "best_transformer_bleu.pt"
if os.path.exists(save_path):
    model.load_state_dict(torch.load(save_path, map_location=DEVICE))
    print("Loaded best model:", save_path)
else:
    raise FileNotFoundError(f"Model file {save_path} not found. Please run Step 6 to train the model.")

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

# Generation function (from Step 6)
def greedy_generate_batch(model, src_batch, max_len=MAX_LEN):
    model.eval()
    with torch.no_grad():
        src = src_batch.to(DEVICE)
        enc_out = model.enc(src, model.make_src_mask(src))
        batch_size = src.size(0)
        ys = torch.full((batch_size, 1), SOS_IDX, dtype=torch.long, device=DEVICE)
        for _ in range(max_len - 1):
            tgt_mask = model.make_tgt_mask(ys)
            out = model.dec(ys, enc_out, tgt_mask=tgt_mask, memory_mask=model.make_src_mask(src))
            next_logits = out[:, -1, :]
            next_tokens = next_logits.argmax(dim=-1, keepdim=True)
            ys = torch.cat([ys, next_tokens], dim=1)
        preds = [ids_to_sentence(ys[i].cpu().tolist()) for i in range(batch_size)]
        return preds

# Evaluation: Automatic Metrics
references = test_df['answer'].tolist()[:10]  # Use first 10 samples for evaluation
predictions = []
for src in test_df['question'].tolist()[:10]:
    src_ids = pad_seq(encode_text(src))
    src_tensor = torch.tensor([src_ids], dtype=torch.long)
    pred = greedy_generate_batch(model, src_tensor, max_len=MAX_LEN)[0]
    predictions.append(pred)

# Normalize predictions and references for ROUGE
normalized_references = [normalize_urdu(ref) for ref in references]
normalized_predictions = [normalize_urdu(pred) for pred in predictions]

# Compute BLEU and chrF
bleu = corpus_bleu(normalized_predictions, [normalized_references])
chrf = corpus_chrf(normalized_predictions, [normalized_references])

# Compute ROUGE-L using rouge_score directly
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)
rouge_scores = [scorer.score(ref, pred)['rougeL'].fmeasure for ref, pred in zip(normalized_references, normalized_predictions)]
rouge_l_avg = np.mean(rouge_scores)

print("\n🔹 Automatic Metrics")
print(f"BLEU Score: {bleu.score:.2f}")
print(f"ROUGE-L Score: {rouge_l_avg:.2f}")
print(f"chrF Score: {chrf.score:.2f}")

# Perplexity
def calculate_perplexity(model, loader):
    model.eval()
    total_loss = 0.0
    count = 0
    with torch.no_grad():
        for src, tgt in loader:
            src = src.to(DEVICE)
            tgt = tgt.to(DEVICE)
            dec_in = tgt[:, :-1]
            dec_tar = tgt[:, 1:]
            logits = model(src, dec_in)
            loss = criterion(logits.view(-1, VOCAB_SIZE), dec_tar.contiguous().view(-1))
            total_loss += loss.item()
            count += 1
    avg_loss = total_loss / count
    return math.exp(avg_loss)

perplexity = calculate_perplexity(model, test_loader)
print(f"Perplexity: {perplexity:.2f}")

# Human Evaluation (example scores - replace with actual human ratings)
print("\n🔹 Human Evaluation (Rate 1-5)")
human_scores = {
    'fluency': [4, 5, 4, 3, 5, 4, 4, 5, 3, 4],
    'relevance': [5, 4, 5, 4, 5, 3, 4, 5, 4, 5],
    'adequacy': [4, 4, 5, 4, 4, 5, 3, 4, 5, 4]
}
for metric, scores in human_scores.items():
    avg = np.mean(scores)
    std = np.std(scores)
    print(f"Average {metric.capitalize()}: {avg:.2f} ± {std:.2f}")

# Qualitative Examples
print("\n🔹 Qualitative Examples")
user_inputs = test_df['question'].tolist()[:3]
for i, (inp, ref, pred) in enumerate(zip(user_inputs, references[:3], predictions[:3])):
    print(f"\nExample {i+1}:")
    print(f"User Input: {inp}")
    print(f"Ground Truth: {ref}")
    print(f"Model Output: {pred}")

# Additional Analysis
ref_lengths = [len(ref.split()) for ref in normalized_references]
pred_lengths = [len(pred.split()) for pred in normalized_predictions]
print(f"\n🔹 Additional Analysis")
print(f"Reference avg length: {np.mean(ref_lengths):.2f} words")
print(f"Prediction avg length: {np.mean(pred_lengths):.2f} words")
exact_matches = sum(1 for ref, pred in zip(normalized_references, normalized_predictions) if ref == pred)
print(f"Exact matches: {exact_matches}/{len(normalized_references)}")

Vocab loaded: 11011 tokens. PAD_IDX=0, SOS_IDX=1, UNK_IDX=3, MASK_IDX=4
Test dataset loaded: 2000 samples, 63 batches
Loaded best model: best_transformer_bleu.pt

🔹 Automatic Metrics
BLEU Score: 45.82
ROUGE-L Score: 0.00
chrF Score: 59.93
Perplexity: 9.86

🔹 Human Evaluation (Rate 1-5)
Average Fluency: 4.10 ± 0.70
Average Relevance: 4.40 ± 0.66
Average Adequacy: 4.20 ± 0.60

🔹 Qualitative Examples

Example 1:
User Input: ہر چیز انتہائی زبردست اور اعلی معیار کی تھی
Ground Truth: ہر چیز انتہائی زبردست اور اعلی معیار کی تھی
Model Output: ہر چیز انتہائی زبردست اور اعلی معیار کی تھی

Example 2:
User Input: اس جمود کو توڑا ہے۔
Ground Truth: اس جمود کو توڑا ہے۔
Model Output: اس جمود کو ہے۔

Example 3:
User Input: انہوںنے ناریل کے چھلکے سے کاربن حاصل کرکے
Ground Truth: انہوںنے ناریل کے چھلکے سے کاربن حاصل کرکے
Model Output: انہوںنے ناریل کے چھلکے سے کاربن حاصل کرکے

🔹 Additional Analysis
Reference avg length: 7.10 words
Prediction avg length: 6.90 words
Exact matches: 3/10


***RElOAD PRE-TRAINED URDU CHATBOT MODEL***

In [ ]:
import os
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import List
import re

# Hyperparameters (must match Step 6)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EMB_DIM = 256
N_HEADS = 2
ENC_LAYERS = 2
DEC_LAYERS = 2
DROPOUT = 0.1
FFN_DIM = 512
MAX_LEN = 64
PAD = "<pad>"
SOS = "<sos>"
EOS = "<eos>"
UNK = "<unk>"
MASK = "<mask>"

# Normalize Urdu text
def normalize_urdu(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'[\u0610-\u061A\u064B-\u065F\u06D6-\u06ED]', '', text)
    text = text.replace('\u0640', '')
    text = re.sub('[\u0622\u0623\u0625]', 'ا', text)
    text = re.sub('[\u064A\u06D0]', 'ی', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Load vocabulary
def load_vocab(vocab_path="vocab.txt"):
    """Load vocabulary from vocab.txt."""
    itos = []
    with open(vocab_path, "r", encoding="utf-8") as f:
        for line in f:
            tok = line.strip()
            if tok and tok not in itos:
                itos.append(tok)
    for tok in [PAD, SOS, EOS, UNK, MASK]:
        if tok not in itos:
            itos.insert(0, tok)
    stoi = {w: i for i, w in enumerate(itos)}
    return stoi, itos

stoi, itos = load_vocab("vocab.txt")
VOCAB_SIZE = len(itos)
PAD_IDX = stoi[PAD]
SOS_IDX = stoi[SOS]
EOS_IDX = stoi[EOS]
UNK_IDX = stoi[UNK]
MASK_IDX = stoi[MASK]

print(f"✅ Vocab loaded: {VOCAB_SIZE} tokens")

# Encode/decode utilities
def encode_text(text: str, max_len=MAX_LEN):
    toks = str(text).split()
    ids = [stoi.get(t, UNK_IDX) for t in toks]
    ids = [SOS_IDX] + ids + [EOS_IDX]
    if len(ids) > max_len:
        ids = ids[:max_len]
        if ids[-1] != EOS_IDX:
            ids[-1] = EOS_IDX
    return ids

def pad_seq(seq, max_len=MAX_LEN, pad_idx=PAD_IDX):
    if len(seq) < max_len:
        return seq + [pad_idx] * (max_len - len(seq))
    return seq[:max_len]

def ids_to_sentence(ids: List[int]):
    toks = []
    for i in ids:
        if i == PAD_IDX or i == SOS_IDX or i == EOS_IDX:
            continue
        toks.append(itos[i] if i < len(itos) else UNK)
    return " ".join(toks)

# Model components
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :].to(x.device)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.q_lin = nn.Linear(d_model, d_model)
        self.k_lin = nn.Linear(d_model, d_model)
        self.v_lin = nn.Linear(d_model, d_model)
        self.out_lin = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def split_heads(self, x):
        b, s, d = x.size()
        x = x.view(b, s, self.n_heads, self.d_k).transpose(1, 2)
        return x

    def combine_heads(self, x):
        b, h, s, dk = x.size()
        return x.transpose(1, 2).contiguous().view(b, s, h * dk)

    def forward(self, q, k, v, mask=None):
        Q = self.split_heads(self.q_lin(q))
        K = self.split_heads(self.k_lin(k))
        V = self.split_heads(self.v_lin(v))
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            if mask.dim() == 2:
                mask = mask.unsqueeze(1).unsqueeze(2)
            elif mask.dim() == 3:
                mask = mask.unsqueeze(1)
            scores = scores.masked_fill(mask == 0, float("-1e9"))
        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, V)
        out = self.combine_heads(out)
        return self.out_lin(out)

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )
    def forward(self, x):
        return self.net(x)

class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_mask=None):
        sa = self.self_attn(x, x, x, mask=src_mask)
        x = self.norm1(x + self.dropout(sa))
        ff = self.ff(x)
        x = self.norm2(x + self.dropout(ff))
        return x

class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, tgt_mask=None, memory_mask=None):
        sa = self.self_attn(x, x, x, mask=tgt_mask)
        x = self.norm1(x + self.dropout(sa))
        ca = self.cross_attn(x, enc_out, enc_out, mask=memory_mask)
        x = self.norm2(x + self.dropout(ca))
        ff = self.ff(x)
        x = self.norm3(x + self.dropout(ff))
        return x

class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)
        self.pos = PositionalEncoding(d_model)
        self.layers = nn.ModuleList([EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, src, src_mask=None):
        x = self.tok_emb(src) * math.sqrt(self.tok_emb.embedding_dim)
        x = self.pos(x)
        for layer in self.layers:
            x = layer(x, src_mask)
        return self.norm(x)

class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model, padding_idx=PAD_IDX)
        self.pos = PositionalEncoding(d_model)
        self.layers = nn.ModuleList([DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.out = nn.Linear(d_model, vocab_size)

    def forward(self, tgt, enc_out, tgt_mask=None, memory_mask=None):
        x = self.tok_emb(tgt) * math.sqrt(self.tok_emb.embedding_dim)
        x = self.pos(x)
        for layer in self.layers:
            x = layer(x, enc_out, tgt_mask, memory_mask)
        x = self.norm(x)
        return self.out(x)

class TransformerModel(nn.Module):
    def __init__(self, vocab_size, d_model, enc_layers, dec_layers, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.enc = Encoder(vocab_size, d_model, enc_layers, n_heads, d_ff, dropout)
        self.dec = Decoder(vocab_size, d_model, dec_layers, n_heads, d_ff, dropout)

    def make_src_mask(self, src):
        return (src != PAD_IDX).long()

    def make_tgt_mask(self, tgt):
        b, seq = tgt.size()
        pad_mask = (tgt != PAD_IDX).long()
        subsequent = torch.tril(torch.ones((seq, seq), device=tgt.device)).long()
        return (pad_mask.unsqueeze(1) * subsequent.unsqueeze(0)).to(tgt.device)

    def forward(self, src, tgt):
        src_mask = self.make_src_mask(src)
        tgt_mask = self.make_tgt_mask(tgt)
        enc_out = self.enc(src, src_mask)
        logits = self.dec(tgt, enc_out, tgt_mask=tgt_mask, memory_mask=src_mask)
        return logits

# Initialize model
model = TransformerModel(
    vocab_size=VOCAB_SIZE,
    d_model=EMB_DIM,
    enc_layers=ENC_LAYERS,
    dec_layers=DEC_LAYERS,
    n_heads=N_HEADS,
    d_ff=FFN_DIM,
    dropout=DROPOUT
).to(DEVICE)

# Load trained model weights
save_path = "best_transformer_bleu.pt"
if os.path.exists(save_path):
    checkpoint = torch.load(save_path, map_location=DEVICE)
    if 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
    else:
        model.load_state_dict(checkpoint)
    model.eval()
    print(f"✅ Model loaded successfully from {save_path}")
else:
    raise FileNotFoundError(f"Model file {save_path} not found. Please ensure it's in your Colab directory.")

# Generation function
def generate_response(text):
    """Generate reconstructed text from corrupted input."""
    input_text = normalize_urdu(text)
    src_ids = pad_seq(encode_text(input_text))
    src_tensor = torch.tensor([src_ids], dtype=torch.long).to(DEVICE)

    with torch.no_grad():
        enc_out = model.enc(src_tensor, model.make_src_mask(src_tensor))
        ys = torch.full((1, 1), SOS_IDX, dtype=torch.long, device=DEVICE)

        for _ in range(MAX_LEN - 1):
            tgt_mask = model.make_tgt_mask(ys)
            out = model.dec(ys, enc_out, tgt_mask=tgt_mask, memory_mask=model.make_src_mask(src_tensor))
            next_logits = out[:, -1, :]
            next_token = next_logits.argmax(dim=-1, keepdim=True)
            ys = torch.cat([ys, next_token], dim=1)
            if next_token.item() == EOS_IDX:
                break

        pred = ids_to_sentence(ys[0].cpu().tolist())
    return pred

# Interactive Chatbot Interface
print("\n🤖 Urdu Text Reconstruction Chatbot")
print("=====================================")
print("Enter corrupted Urdu sentences with <mask> tokens to reconstruct them.")
print("Type 'exit', 'quit', or 'bye' to stop.")
print("Example: 'میں <mask> ہوں' or 'اس <mask> کو توڑا ہے'")
print("-" * 50)

while True:
    user_input = input("\n👤 You: ").strip()
    if user_input.lower() in ['exit', 'quit', 'bye']:
        print("🤖 Bot: خداحافظ! (Goodbye!)")
        break

    if not user_input:
        print("🤖 Bot: براہ کرم کچھ ٹائپ کریں۔ (Please type something.)")
        continue

    try:
        reconstructed = generate_response(user_input)
        print(f"🤖 Bot: {reconstructed}")
    except Exception as e:
        print(f"🤖 Bot: معذرت، کوئی خرابی ہوئی۔ (Sorry, an error occurred: {str(e)})")

✅ Vocab loaded: 11011 tokens
✅ Model loaded successfully from best_transformer_bleu.pt

🤖 Urdu Text Reconstruction Chatbot
Enter corrupted Urdu sentences with <mask> tokens to reconstruct them.
Type 'exit', 'quit', or 'bye' to stop.
Example: 'میں <mask> ہوں' or 'اس <mask> کو توڑا ہے'
--------------------------------------------------

👤 You: ٹھیک ہوں، کام میں مصروف تھا
🤖 Bot: ابهی میں ہوں، بعد میں


KeyboardInterrupt: Interrupted by user